In [ ]:
# importations
import sys
import pandas as pd

import os
import json 
from pathlib import Path

# Access packages from the root
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..")))

from app.services.quiz_router import handle_quiz_request
from app.agent.rag.ingestion.data_ingestion import process_and_load_file
from app.agent.rag.ingestion.embeddings import get_embedding_model, generate_embeddings
from app.agent.rag.ingestion.vector_store import get_vector_store, add_documents_to_chroma
from app.agent.rag.chains.notes_chain import run_notes_chain
from app.agent.rag.retrieval.retriever import get_semantic_chunks, get_topic_chunks
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from typing import Dict, List
from langchain_google_genai import ChatGoogleGenerativeAI
from app.agent.rag.chains.qa_chain import run_qa_chain
from app.agent.rag.chains.eval_chain import run_eval_chain


In [4]:

load_dotenv()
api_key = os.environ.get('GOOGLE_API_KEY')

# ADC together with an API key for it to work
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    project="denseless",
    location="us-central1",
    vertexai=True,
)

json_eval_dataset = {}



INFO:google_genai._api_client:The user provided project/location will take precedence over the Vertex AI API key from the environment variable.


In [6]:
# Define the LTM of the hypothetical student
### Done inside its LTM

# Ingestion of note of interest
topic = "Disaster Recovery Plan"
docs = process_and_load_file(f"../pdfs/{topic}.pdf")
embedder = get_embedding_model()
vectors = generate_embeddings(docs, embedder)
store = get_vector_store(student_id="1019", embedder=embedder)
add_documents_to_chroma(store, vectors, docs, False, "CSU", topic, topic)


INFO:unstructured-client:split_pdf event=plan_created operation_id=5e62f549-29a6-4313-83ac-fa6824e602f3 filename=Disaster Recovery Plan.pdf strategy=hi_res page_range=1-4 page_count=4 split_size=2 chunk_count=2 concurrency=5 allow_failed=False cache_mode=disabled timeout_seconds=None retry_config_mode=sdk_default_or_unset


PDF page count: 4
Processing: Disaster Recovery Plan.pdf


INFO:httpx:HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO:unstructured-client:split_pdf event=batch_start operation_id=5e62f549-29a6-4313-83ac-fa6824e602f3 chunk_count=2 concurrency=5 allow_failed=False client_timeout_seconds=None future_timeout_seconds=3605 num_waves=1
INFO:httpx:HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO:unstructured-client:split_pdf event=batch_complete operation_id=5e62f549-29a6-4313-83ac-fa6824e602f3 chunk_count=2 success_count=2 failure_count=0 transport_failure_count=0 elapsed_ms=32672 allow_failed=False
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5


  → Parsed into 49 document(s)
Loading embedding model: BAAI/bge-small-en-v1.5 (device=cpu)


INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BA

  → Embedding model loaded successfully. Embedding dimension: 384
Generating embeddings for 49 document chunk(s)...


Embedding documents: 100%|██████████| 2/2 [00:01<00:00,  1.30it/s]


  → Successfully generated 49 embedding vectors.
  → Embeddings shape: (49, 384)
Vector store backend: 'chroma' | student: '1019'
Initialising Chroma store — student: '1019'
  → Collection : student_1019
  → Persist dir: ../chroma_db
  → Chroma store ready. 
Documents in collection: 401
  [vector_store] 401 existing chunk(s) found in collection.
  [vector_store] 49 duplicate chunk(s) skipped.
  [vector_store] ✓ All documents already exist in store — nothing to add.


In [7]:
# Ask questions to the student then it uses the relevant ltm context
questions = [
    "What is a Disaster Recovery Plan?",
    "What is disaster recovery?",
    "What is an RTO?",
    "What is an RPO?",
    "What is a hot site?",
    "What is a cold site?",
    "What is a warm site?",
    "What is a disaster recovery site?",
    "What are the three elements of an effective DR plan?",
    "What types of events are considered disasters in a DRP context?",
    "What is a business continuity plan?",
    "What is the IT inventory chapter in a DRP?",
    "What are backup procedures in a DRP?",
    "What is the restoration chapter in a DRP?",
    "What are corrective DR measures?",
    "What are preventive DR measures?",
    "What are detective DR measures?",
    "What is the difference between RTO and RPO?",
    "What is the difference between a DRP and a BCP?",
    "What is the difference between a cold site and a warm site?",
    "What is the difference between a warm site and a hot site?",
    "How does a cold site compare to a hot site in terms of recovery readiness?",
    "What distinguishes detective measures from corrective measures in DR?",
    "What distinguishes preventive measures from detective measures in DR?",
    "How does RPO differ from RTO in terms of what they measure?",
    "How do you create a disaster recovery team?",
    "How do you identify and assess disaster risks?",
    "How do you determine which applications are critical in a DRP?",
    "How do you specify backup procedures in a DRP?",
    "How do you test a DRP?",
    "How do you maintain a DRP over time?",
    "How does RPO help define backup frequency?",
    "How do you recover from a complete systems loss?",
    "How does a hot site work when disaster strikes?",
    "How do you determine an acceptable RTO for a system?",
    "What should be prioritised when a ransomware attack hits critical infrastructure?",
    "What happens to an organisation with no DRP during a cyberattack?",
    "How would you set an RTO for a payroll processing system?",
    "What documents should be stored at an off-site location?",
    "What critical supplies should be kept off-site in a DRP?",
    "How would you handle a natural disaster that destroys the primary data center?",
    "When would an organisation choose a warm site over a hot site?",
    "What is the first action when a disaster is detected?",
    "How does data replication factor into a hot site strategy?",
    "Why should all employees understand the DRP, not just the recovery team?",
    "What does the goals chapter of a DRP specifically define?",
    "What does the personnel chapter of a DRP cover?",
    "What makes a hot site the ideal DR site?",
    "Why is a cold site limiting to a business during recovery?",
    "Why is disaster recovery planning described as a continual process rather than a one-time exercise?",
]

ltm_memories = (
    "The student dislikes analogies and finds them condescending. They prefer straight to the point, technical explanations without illustrative comparisons.;"
    " The student prefers concise responses. They get disengaged when answers are padded with background context they did not ask for.;"
    " The student consistently confuses RTO (Recovery Time Objective) and RPO (Recovery Point Objective). They have mixed them up across multiple past interactions."
)

# Get answer to each question and store in dict
for i, question in enumerate(questions, start=1):
    key = str(i).zfill(3)
    response = run_qa_chain(
        "student_1019",
        question,
        f"PQ_chat_{i}",
        False,
        store,
        llm,
        "fast",
        "Disaster Recovery Plan.pdf",
        "PQ",
    )
    answer = response.content["answer"]

    json_eval_dataset[key] = {
        "student_question": question,
        "ltm_memories"    : ltm_memories,
        "qa_output"       : answer,
    }

INFO:google_genai.models:AFC is enabled with max remote calls: 10.


  [token_service] Tokens remaining for 'student_1019': 9,519,766
  [token_guard] Checking tokens — student: student_1019 | remaining: 9519766 | chain: run_qa_chain
[qa_chain] Starting QA — student: 'student_1019' | topic: 'Disaster Recovery Plan.pdf' | convo: 'PQ_chat_1' | pace: 'fast'
[qa_chain] STM loaded for 'PQ_chat_1' (1 turns summarised).
[qa_chain] STM loaded for 'PQ_chat_1' (1 turns summarised).
[qa_chain] STM context loaded (621 chars).
[qa_chain] LTM store loaded for student 'student_1019' (3 memories).
[qa_chain] LTM: 1 relevant memory/memories retrieved.


INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC is enabled with max remote calls: 10.


[qa_chain] Question reformulated.
[Retriever] Phase 1 — Semantic search (top_k=5, threshold=0.4): 'What is a Disaster Recovery Plan'
Querying Chroma (top_k=5, threshold=0.4): 'Represent this sentence for searching relevant passages: Wha...'
  → Retrieved 5 chunk(s), 5 passed threshold.
[Retriever] → 5 seed chunk(s) passed threshold.
[Retriever] Phase 2 — Section context expansion...
[Retriever] Expanding context across 1 section(s): {'What is a Disaster Recovery Plan'}
[Retriever]   → Section 'What is a Disaster Recovery Plan': 5 chunk(s) fetched.
[Retriever] → Expansion complete: 5 seed → 5 unique chunk(s) after deduplication.
[Retriever] → Final result: 5 chunk(s) returned to chain.
[qa_chain] Retrieved 5 chunk(s).
1. What is a Disaster Recovery Plan?
2. Disaster recovery (DR) is an organization’s ability to restore access and functionality to IT infrastructure after a disaster event, whether natural or caused by human action (or error). DR is considered a subset of business continui

INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC is enabled with max remote calls: 10.


...

[qa_chain] QA LLM call succeeded. Grounded: None


INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[qa_chain] STM summary updated (308 chars).
[qa_chain] STM saved for 'PQ_chat_50' (1 turns summarised).


INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[qa_chain] LTM raw manager output: [ExtractedMemory(id='209ad179-cadf-4957-9b7c-1b593ffe518a', content=Memory(content="The student is asking for explanations of concepts rather than demonstrating their own understanding or misconceptions. Therefore, no new memories about the student's learning state can be extracted from this exchange."))]
[qa_chain] LTM store saved for student 'student_1019' (10 memories).
[qa_chain] LTM: 1 new memory/memories extracted and saved.
[qa_chain] QA chain complete for student 'student_1019'.
  [token_service] Deducted 1,312 tokens — student: 'student_1019' | chain: run_qa_chain | in: 1,214 | out: 98 | remaining: 9,475,504
  [token_guard] ✓ Tokens deducted — student: student_1019 | consumed: 1312 | remaining: 9475504


In [8]:
# Export to json
output_path = Path(os.getcwd()) / "rag" / "eval_data" / "pq_eval_data.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(json_eval_dataset, f, indent=4, ensure_ascii=False)

print(f"Successfully completed. Exported to {output_path}")

Successfully completed. Exported to c:\Users\USER\Documents\AI projects\DenseLess\app\agent\rag\eval_data\pq_eval_data.json
